# Gemma-12B: Neutral-topic follow-up pilot

Follow-up to the climate generalization pilot (`experiments/e1_climate/gemma4-12b/`), which found
Gemma-12B *inverts* on that pilot: it prefers the LOWER-engagement post in all 21 of 21 tested
off-diagonal scale-pairs, the opposite of every other model's normal conformity direction on the
same pilot, and the opposite of Gemma-12B's own behavior on the main Pop-vs-Latin study. Diagonal
(tied-engagement) accuracy there is 74.3%, and refusals are near-zero (0.2%), so the inversion is
real, not a parsing artifact -- confirmed 2026-08-28 following the supervisor's request to triple-check it.

This pilot asks whether that inversion is specific to the climate/renewable-energy content, or a
general property of this model's behavior on this chart-plus-engagement format regardless of topic.
Same protocols as the climate pilot (baseline single-image like/scroll + logprobs, single-image
across the 6 `metrics/realistic` engagement scales + logprobs, full 7x7 paired A/B `metrics` grid),
same underlying fabricated data (byte-identical numeric series, verified against the climate pilot's
own generator for all 25 posts), only the topic swapped to a maximally generic, content-free frame
("Product A" vs. "Product B" quarterly revenue) -- pointed at `neutral_pilot/posts/`.

**25 posts (not 50/100): this is a scoped pilot, not a full replication. Gemma-12B only, per the
supervisor's request -- not rolled out to the other 5 models.**

In [ ]:
import sys, subprocess

# 1. Uninstall torchaudio
subprocess.run([sys.executable, "-m", "pip", "uninstall", "torchaudio", "-y"])

# 2. Install PyTorch with CUDA 12.4
subprocess.run([sys.executable, "-m", "pip", "install",
    "torch==2.6.0", "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
    "--user", "-q"], check=True)

# 3. Install latest transformers and accelerate (allowing pip to pull compatible tokenizers naturally)
subprocess.run([sys.executable, "-m", "pip", "install",
    "git+https://github.com/huggingface/transformers",
    "accelerate",
    "--user", "-q"], check=True)

print("✅ Installation complete — restart the kernel now")



Restart kernel after running the setup cell above.

In [ ]:
!nvidia-smi

In [ ]:
# --- HF Auth ---
import sys
sys.path.append("/home/jovyan")
from config_hf_token import HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN)

# --- Path setup ---
from pathlib import Path
ROOT_DIR = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR / "experiments/e1"))

# --- Load Gemma model ---
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/gemma-4-12B-it"
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, padding_side="left")
device = model.device

In [ ]:
from e1_utils.sampling import build_paired_sample
from e1_utils.e1_optimized import (
    LIKE_PROMPT_SINGLE, LIKE_PROMPT_YESNO, LIKE_PROMPT_PAIR, ADJACENT_PAIRS,
    run_e1_baseline, run_e1_metrics, run_e1_metrics_paired
)
from e1_utils.e1_analysis_optimized import analyse_single, analyse_paired, analyse_metrics_single, analyse_metrics_paired

# --- Configuration: neutral-topic follow-up pilot, NOT the main benchmarking/ pool ---
EXPERIMENT_DIR = Path().resolve().parent        # experiments/e1_neutral/  -- Gemma-12B only for this pilot
OUTPUT_DIR = Path().resolve() / "outputs"
SEED = 42
SAMPLE_SIZE = 25

correct_dir = ROOT_DIR / "neutral_pilot/posts/correct/PNGs"
incorrect_dir = ROOT_DIR / "neutral_pilot/posts/incorrect/PNGs"
all_images = build_paired_sample(correct_dir, incorrect_dir, SEED, SAMPLE_SIZE, EXPERIMENT_DIR)
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]

# metrics/realistic condition only -- the condition that showed the strongest conformity effect
# in the main study (Section 6.2), and the one the climate-pilot inversion was found under
correct_base = ROOT_DIR / "neutral_pilot/posts/correct/PNGs/metrics/realistic"
incorrect_base = ROOT_DIR / "neutral_pilot/posts/incorrect/PNGs/metrics/realistic"


In [ ]:
from e1_utils.inference_gemma import run_inference_gemma

In [ ]:
from e1_utils.e1_optimized import (
    run_e1_baseline_logprobs, run_e1_metrics_logprobs,
    LIKE_CANDIDATES_SINGLE, LIKE_CANDIDATES_YESNO
)

from e1_utils.inference_gemma import run_inference_with_scores_gemma

## Approach 1 -- single image, like/scroll, baseline (0 engagement)

In [ ]:
run_e1_baseline(all_images, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_baseline.json",
              inference_fn=run_inference_gemma)

In [ ]:
run_e1_baseline_logprobs(all_images, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, candidates=LIKE_CANDIDATES_SINGLE,
              output_filename="e1_results_baseline_logprobs.json", score_fn=run_inference_with_scores_gemma)

In [ ]:
analyse_single(OUTPUT_DIR, "e1_results_baseline.json", like_answer="like")

## Approach 1 variant -- single image, like/scroll, across the 6 `metrics/realistic` engagement scales

In [ ]:
run_e1_metrics(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_metrics.json",
              inference_fn=run_inference_gemma)

In [ ]:
run_e1_metrics_logprobs(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, candidates=LIKE_CANDIDATES_SINGLE,
              output_filename="e1_results_metrics_logprobs.json", score_fn=run_inference_with_scores_gemma)

In [ ]:
analyse_metrics_single(OUTPUT_DIR, "e1_results_metrics.json", like_answer="like")

## Approach 2 -- paired A/B forced choice, full 7x7 `metrics/realistic` disparity grid

In [ ]:
import time
start = time.time()
run_e1_metrics_paired(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR, SEED,
              prompt=LIKE_PROMPT_PAIR, output_filename="e1_results_metrics_paired.json",
              inference_fn=run_inference_gemma)
elapsed = time.time() - start
print(f"\n⏱ Total runtime: {elapsed/60:.1f} min")

In [ ]:
analyse_metrics_paired(OUTPUT_DIR, "e1_results_metrics_paired.json")